# Dataset Summary

## Dataset Source

The selected dataset consists of experimental Li-ion battery aging data containing charge, discharge, and impedance measurements. The dataset contains four batteries:

- B0005
- B0006
- B0007
- B0018

## Number of Batteries

4 batteries.

## Operation Records

The dataset contains three types of operations:

- Charge
- Discharge
- Impedance

| Battery | Total Operations | Charge | Discharge | Impedance |
|---------|-----------------:|-------:|----------:|----------:|
| B0005 | 616 | 170 | 168 | 278 |
| B0006 | 616 | 170 | 168 | 278 |
| B0007 | 616 | 170 | 168 | 278 |
| B0018 | 319 | 134 | 132 | 53 |
| **Total** | **2167** | **644** | **636** | **887** |

## Discharge Data

Discharge operations contain the following measurements:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_load`
- `Voltage_load`
- `Time`
- `Capacity`

All **636 discharge operations** contain a valid capacity measurement.

The measured capacity generally decreases over repeated discharge operations, indicating battery degradation with aging.

The observed initial and final discharge capacities are:

| Battery | Initial Capacity (Ah) | Final Capacity (Ah) | Minimum Capacity (Ah) |
|---------|----------------------:|--------------------:|----------------------:|
| B0005 | 1.8565 | 1.3251 | 1.2875 |
| B0006 | 2.0353 | 1.1857 | 1.1538 |
| B0007 | 1.8911 | 1.4325 | 1.4005 |
| B0018 | 1.8550 | 1.3411 | 1.3411 |

## Sampling Information

The time-series measurements are sampled at non-uniform time intervals.

The number of samples per discharge operation varies between operations:

| Battery | Minimum Samples | Maximum Samples | Average Samples |
|---------|----------------:|----------------:|----------------:|
| B0005 | 179 | 371 | 299.32 |
| B0006 | 179 | 371 | 299.32 |
| B0007 | 179 | 371 | 299.32 |
| B0018 | 200 | 366 | 264.14 |

Therefore, the raw discharge measurements are variable-length time series and will require appropriate standardization during preprocessing.

## Missing Values

No missing numerical values were detected in the charge, discharge, or impedance data of B0005, B0006, and B0007.

B0018 contains **6 NaN values in its charge operations**. No NaN values were found in its discharge or impedance operations.

These missing values will be investigated and handled during preprocessing. No data will be removed without documenting the reason.

## Data Quality Checks

The following checks were performed:

- No empty discharge time sequences were found.
- No non-monotonic discharge time sequences were found.
- No duplicate timestamps were found in discharge operations.
- All 636 discharge operations contain a valid capacity measurement.
- Discharge measurements are stored as numerical floating-point values.
- Discharge operations contain variable-length time-series measurements.
- Capacity values show an overall decreasing trend across battery aging.

## Relevant Battery Parameters

The relevant parameters identified in the dataset include:

- Battery ID
- Operation type
- Time
- Voltage
- Current
- Temperature
- Capacity
- Battery impedance
- Rectified impedance
- Electrolyte resistance (`Re`)
- Charge transfer resistance (`Rct`)

The final set of features to be included in the standardized dataset will be finalized in consultation with the other project members.

## Cycle Definition

The raw dataset contains charge, discharge, and impedance operations in a non-uniform sequence. Therefore, a cycle should not be identified simply by grouping every fixed number of raw records.

For the current analysis, discharge operations are treated as the primary cycle-level records because each discharge operation contains a measured capacity value. The final cycle definition and SOH formulation will be confirmed with the project team before the standardized dataset is created.

## Initial Dataset Findings — Battery B0005

The B0005 MATLAB file contains 616 operation records consisting of:

- 170 charge operations
- 168 discharge operations
- 278 impedance operations

Each operation contains four top-level fields:

- `type`
- `ambient_temperature`
- `time`
- `data`

For charge operations, the measurement fields are:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_charge`
- `Voltage_charge`
- `Time`

For discharge operations, the observed fields in B0005 are:

- `Voltage_measured`
- `Current_measured`
- `Temperature_measured`
- `Current_load`
- `Voltage_load`
- `Time`
- `Capacity`

The first discharge operation has a measured capacity of approximately 1.8565 Ah and contains 197 samples for each time-series measurement.

The discharge current is approximately -2 A after the initial samples, while battery voltage decreases and temperature increases during the discharge operation.

In [44]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    counts = Counter()
    nan_counts = Counter()

    for i in range(cycle_data.shape[1]):

        cycle_type = cycle_data[0, i]["type"].item()
        counts[cycle_type] += 1

        data = cycle_data[0, i]["data"][0, 0]

        for field in data.dtype.names:
            values = np.asarray(data[field]).flatten()

            if np.issubdtype(values.dtype, np.number):
                nan_counts[cycle_type] += np.isnan(values).sum()

    print(f"\n{battery_name}")
    print("-" * 30)

    for operation in ["charge", "discharge", "impedance"]:
        print(
            f"{operation:10s}: "
            f"{counts[operation]} operations, "
            f"{nan_counts[operation]} NaN values"
        )


B0005
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0006
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0007
------------------------------
charge    : 170 operations, 0 NaN values
discharge : 168 operations, 0 NaN values
impedance : 278 operations, 0 NaN values

B0018
------------------------------
charge    : 134 operations, 6 NaN values
discharge : 132 operations, 0 NaN values
impedance : 53 operations, 0 NaN values


In [43]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    sample_counts = []

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()
            sample_counts.append(len(time_values))

    print(f"\n{battery_name}")
    print("-" * 30)
    print("Number of discharge operations:", len(sample_counts))
    print("Minimum samples:", min(sample_counts))
    print("Maximum samples:", max(sample_counts))
    print("Average samples:", round(np.mean(sample_counts), 2))


B0005
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0006
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0007
------------------------------
Number of discharge operations: 168
Minimum samples: 179
Maximum samples: 371
Average samples: 299.32

B0018
------------------------------
Number of discharge operations: 132
Minimum samples: 200
Maximum samples: 366
Average samples: 264.14


In [42]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    discharge_count = 0
    missing_capacity = 0

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            discharge_count += 1

            data = cycle_data[0, i]["data"][0, 0]
            capacity = np.asarray(data["Capacity"]).flatten()

            if len(capacity) == 0 or np.isnan(capacity[0]):
                missing_capacity += 1

    print(f"{battery_name}:")
    print(f"  Discharge operations: {discharge_count}")
    print(f"  Missing capacity: {missing_capacity}")

B0005:
  Discharge operations: 168
  Missing capacity: 0
B0006:
  Discharge operations: 168
  Missing capacity: 0
B0007:
  Discharge operations: 168
  Missing capacity: 0
B0018:
  Discharge operations: 132
  Missing capacity: 0


In [41]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    types = [
        cycle_data[0, i]["type"].item()
        for i in range(cycle_data.shape[1])
    ]

    print(f"\n{battery_name}")
    print("-" * 30)

    # Show the first 50 operations with their indices
    for i, operation in enumerate(types[:50]):
        print(i, operation)


B0005
------------------------------
0 charge
1 discharge
2 charge
3 discharge
4 charge
5 discharge
6 charge
7 discharge
8 charge
9 discharge
10 charge
11 discharge
12 charge
13 discharge
14 charge
15 discharge
16 charge
17 discharge
18 charge
19 discharge
20 charge
21 discharge
22 charge
23 charge
24 discharge
25 charge
26 discharge
27 charge
28 discharge
29 charge
30 discharge
31 charge
32 discharge
33 charge
34 discharge
35 charge
36 discharge
37 charge
38 discharge
39 charge
40 impedance
41 discharge
42 impedance
43 charge
44 impedance
45 discharge
46 impedance
47 charge
48 impedance
49 discharge

B0006
------------------------------
0 charge
1 discharge
2 charge
3 discharge
4 charge
5 discharge
6 charge
7 discharge
8 charge
9 discharge
10 charge
11 discharge
12 charge
13 discharge
14 charge
15 discharge
16 charge
17 discharge
18 charge
19 discharge
20 charge
21 discharge
22 charge
23 charge
24 discharge
25 charge
26 discharge
27 charge
28 discharge
29 charge
30 discharge
31 charg

In [40]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    types = [
        cycle_data[0, i]["type"].item()
        for i in range(cycle_data.shape[1])
    ]

    unmatched_discharge = 0

    for i, operation in enumerate(types):
        if operation == "discharge":
            if i == 0 or types[i - 1] != "charge":
                unmatched_discharge += 1

    print(f"{battery_name}:")
    print(f"  Discharges without immediately preceding charge: {unmatched_discharge}")

B0005:
  Discharges without immediately preceding charge: 137
B0006:
  Discharges without immediately preceding charge: 137
B0007:
  Discharges without immediately preceding charge: 137
B0018:
  Discharges without immediately preceding charge: 23


In [39]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    types = []

    for i in range(cycle_data.shape[1]):
        types.append(cycle_data[0, i]["type"].item())

    print(f"\n{battery_name}")
    print("-" * 30)

    # Show the first 20 operations
    print("First 20 operations:")
    print(types[:20])


B0005
------------------------------
First 20 operations:
['charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge']

B0006
------------------------------
First 20 operations:
['charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge']

B0007
------------------------------
First 20 operations:
['charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge', 'charge', 'discharge']

B0018
------------------------------
First 20 operations:
['charge', 'impedance', 'discharge', 'impedance', 'charge', 'impedance'

In [38]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    duplicate_time_cycles = 0
    total_duplicate_timestamps = 0

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()

            duplicates = len(time_values) - len(np.unique(time_values))

            if duplicates > 0:
                duplicate_time_cycles += 1
                total_duplicate_timestamps += duplicates

    print(f"{battery_name}:")
    print(f"  Discharge operations with duplicate timestamps: {duplicate_time_cycles}")
    print(f"  Total duplicate timestamps: {total_duplicate_timestamps}")

B0005:
  Discharge operations with duplicate timestamps: 0
  Total duplicate timestamps: 0
B0006:
  Discharge operations with duplicate timestamps: 0
  Total duplicate timestamps: 0
B0007:
  Discharge operations with duplicate timestamps: 0
  Total duplicate timestamps: 0
B0018:
  Discharge operations with duplicate timestamps: 0
  Total duplicate timestamps: 0


In [37]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    total_discharge = 0
    non_monotonic = 0
    empty_time = 0

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            total_discharge += 1

            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()

            if len(time_values) == 0:
                empty_time += 1
                continue

            if not np.all(np.diff(time_values) >= 0):
                non_monotonic += 1

    print(f"{battery_name}:")
    print(f"  Discharge operations: {total_discharge}")
    print(f"  Non-monotonic time sequences: {non_monotonic}")
    print(f"  Empty time sequences: {empty_time}")

B0005:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0006:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0007:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0018:
  Discharge operations: 132
  Non-monotonic time sequences: 0
  Empty time sequences: 0


In [36]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    total_discharge = 0
    non_monotonic = 0
    empty_time = 0

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            total_discharge += 1

            data = cycle_data[0, i]["data"][0, 0]
            time_values = np.asarray(data["Time"]).flatten()

            if len(time_values) == 0:
                empty_time += 1
                continue

            if not np.all(np.diff(time_values) >= 0):
                non_monotonic += 1

    print(f"{battery_name}:")
    print(f"  Discharge operations: {total_discharge}")
    print(f"  Non-monotonic time sequences: {non_monotonic}")
    print(f"  Empty time sequences: {empty_time}")

B0005:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0006:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0007:
  Discharge operations: 168
  Non-monotonic time sequences: 0
  Empty time sequences: 0
B0018:
  Discharge operations: 132
  Non-monotonic time sequences: 0
  Empty time sequences: 0


In [35]:
for battery_name, battery in all_batteries.items():

    cycles = all_batteries[battery_name][battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    print(f"\n{battery_name}")
    print("-" * 30)

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]

            for field in data.dtype.names:
                values = np.asarray(data[field]).flatten()

                print(
                    f"{field:20s} "
                    f"dtype={values.dtype}, "
                    f"min={values.min():.4f}, "
                    f"max={values.max():.4f}"
                )

            break


B0005
------------------------------
Voltage_measured     dtype=float64, min=2.6125, max=4.1915
Current_measured     dtype=float64, min=-2.0180, max=0.0007
Temperature_measured dtype=float64, min=24.3260, max=38.9822
Current_load         dtype=float64, min=-1.9984, max=-0.0004
Voltage_load         dtype=float64, min=0.0000, max=4.2060
Time                 dtype=float64, min=0.0000, max=3690.2340
Capacity             dtype=float64, min=1.8565, max=1.8565

B0006
------------------------------
Voltage_measured     dtype=float64, min=2.4758, max=4.1798
Current_measured     dtype=float64, min=-2.0155, max=0.0004
Temperature_measured dtype=float64, min=24.2771, max=39.1630
Current_load         dtype=float64, min=-1.9992, max=-0.0006
Voltage_load         dtype=float64, min=0.0000, max=4.1950
Time                 dtype=float64, min=0.0000, max=3690.2340
Capacity             dtype=float64, min=2.0353, max=2.0353

B0007
------------------------------
Voltage_measured     dtype=float64, min=2.14

In [34]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    total_nan = 0
    discharge_count = 0

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            discharge_count += 1
            data = cycle_data[0, i]["data"][0, 0]

            for field in data.dtype.names:
                values = np.asarray(data[field]).flatten()
                total_nan += np.isnan(values).sum()

    print(f"{battery_name}:")
    print(f"  Discharge operations: {discharge_count}")
    print(f"  Total NaN values: {total_nan}")

B0005:
  Discharge operations: 168
  Total NaN values: 0
B0006:
  Discharge operations: 168
  Total NaN values: 0
B0007:
  Discharge operations: 168
  Total NaN values: 0
B0018:
  Discharge operations: 132
  Total NaN values: 0


In [32]:
print("TEST - capacity extraction")

TEST - capacity extraction


In [33]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    capacities = []

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]
            capacity = float(data["Capacity"].flatten()[0])
            capacities.append(capacity)

    print(f"\n{battery_name}")
    print("-" * 20)
    print("Number of discharge capacities:", len(capacities))
    print("First capacity:", capacities[0])
    print("Last capacity:", capacities[-1])
    print("Minimum capacity:", min(capacities))
    print("Maximum capacity:", max(capacities))


B0005
--------------------
Number of discharge capacities: 168
First capacity: 1.8564874208181574
Last capacity: 1.3250793286429356
Minimum capacity: 1.2874525221379407
Maximum capacity: 1.8564874208181574

B0006
--------------------
Number of discharge capacities: 168
First capacity: 2.035337591005598
Last capacity: 1.1856752327929356
Minimum capacity: 1.15381833159625
Maximum capacity: 2.035337591005598

B0007
--------------------
Number of discharge capacities: 168
First capacity: 1.89105229539079
Last capacity: 1.4324552720625434
Minimum capacity: 1.4004552399066514
Maximum capacity: 1.89105229539079

B0018
--------------------
Number of discharge capacities: 132
First capacity: 1.8550045207910817
Last capacity: 1.341051440640485
Minimum capacity: 1.341051440640485
Maximum capacity: 1.8550045207910817


In [28]:
battery_names = ["B0006", "B0007", "B0018"]

batteries = {}

with zipfile.ZipFile(zip_path, "r") as z:
    for name in battery_names:
        with z.open(name + ".mat") as f:
            batteries[name] = loadmat(f)

print("Loaded batteries:")
print(list(batteries.keys()))

Loaded batteries:
['B0006', 'B0007', 'B0018']


In [29]:
from collections import Counter

all_batteries = {
    "B0005": battery_data,
    "B0006": batteries["B0006"],
    "B0007": batteries["B0007"],
    "B0018": batteries["B0018"]
}

for battery_name, battery in all_batteries.items():
    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    cycle_types = []

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()
        cycle_types.append(cycle_type)

    counts = Counter(cycle_types)

    print(f"\n{battery_name}")
    print("-" * 20)
    print("Total operations:", len(cycle_types))
    print("Charge:", counts["charge"])
    print("Discharge:", counts["discharge"])
    print("Impedance:", counts["impedance"])


B0005
--------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0006
--------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0007
--------------------
Total operations: 616
Charge: 170
Discharge: 168
Impedance: 278

B0018
--------------------
Total operations: 319
Charge: 134
Discharge: 132
Impedance: 53


In [30]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    capacities = []

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]
            capacity = float(data["Capacity"].flatten()[0])
            capacities.append(capacity)

    print(f"\n{battery_name}")
    print("-" * 20)
    print("Number of discharge capacities:", len(capacities))
    print("First capacity:", capacities[0])
    print("Last capacity:", capacities[-1])
    print("Minimum capacity:", min(capacities))
    print("Maximum capacity:", max(capacities))


B0005
--------------------
Number of discharge capacities: 168
First capacity: 1.8564874208181574
Last capacity: 1.3250793286429356
Minimum capacity: 1.2874525221379407
Maximum capacity: 1.8564874208181574

B0006
--------------------
Number of discharge capacities: 168
First capacity: 2.035337591005598
Last capacity: 1.1856752327929356
Minimum capacity: 1.15381833159625
Maximum capacity: 2.035337591005598

B0007
--------------------
Number of discharge capacities: 168
First capacity: 1.89105229539079
Last capacity: 1.4324552720625434
Minimum capacity: 1.4004552399066514
Maximum capacity: 1.89105229539079

B0018
--------------------
Number of discharge capacities: 132
First capacity: 1.8550045207910817
Last capacity: 1.341051440640485
Minimum capacity: 1.341051440640485
Maximum capacity: 1.8550045207910817


In [26]:
import numpy as np

print("Time (first 10):")
print(data["Time"].flatten()[:10])

print("\nVoltage (first 10):")
print(data["Voltage_measured"].flatten()[:10])

print("\nCurrent (first 10):")
print(data["Current_measured"].flatten()[:10])

print("\nTemperature (first 10):")
print(data["Temperature_measured"].flatten()[:10])

Time (first 10):
[  0.     16.781  35.703  53.781  71.922  90.094 108.281 126.453 144.641
 162.844]

Voltage (first 10):
[4.19149181 4.19074907 3.97487091 3.95171671 3.93435249 3.92005844
 3.90790351 3.89703572 3.88747658 3.87895859]

Current (first 10):
[-4.90158921e-03 -1.47800555e-03 -2.01252832e+00 -2.01397936e+00
 -2.01114360e+00 -2.01300663e+00 -2.01439966e+00 -2.01160289e+00
 -2.01801469e+00 -2.01313475e+00]

Temperature (first 10):
[24.33003389 24.32599342 24.38908513 24.54475232 24.7313852  24.9098164
 25.10588415 25.31701856 25.5094231  25.70360345]


In [31]:
for battery_name, battery in all_batteries.items():

    cycles = battery[battery_name]["cycle"]
    cycle_data = cycles[0, 0]

    capacities = []

    for i in range(cycle_data.shape[1]):
        cycle_type = cycle_data[0, i]["type"].item()

        if cycle_type == "discharge":
            data = cycle_data[0, i]["data"][0, 0]
            capacity = float(data["Capacity"].flatten()[0])
            capacities.append(capacity)

    print(f"\n{battery_name}")
    print("-" * 20)
    print("Number of discharge capacities:", len(capacities))
    print("First capacity:", capacities[0])
    print("Last capacity:", capacities[-1])
    print("Minimum capacity:", min(capacities))
    print("Maximum capacity:", max(capacities))


B0005
--------------------
Number of discharge capacities: 168
First capacity: 1.8564874208181574
Last capacity: 1.3250793286429356
Minimum capacity: 1.2874525221379407
Maximum capacity: 1.8564874208181574

B0006
--------------------
Number of discharge capacities: 168
First capacity: 2.035337591005598
Last capacity: 1.1856752327929356
Minimum capacity: 1.15381833159625
Maximum capacity: 2.035337591005598

B0007
--------------------
Number of discharge capacities: 168
First capacity: 1.89105229539079
Last capacity: 1.4324552720625434
Minimum capacity: 1.4004552399066514
Maximum capacity: 1.89105229539079

B0018
--------------------
Number of discharge capacities: 132
First capacity: 1.8550045207910817
Last capacity: 1.341051440640485
Minimum capacity: 1.341051440640485
Maximum capacity: 1.8550045207910817


In [25]:
data = discharge_cycle["data"][0, 0]

print("Measurement fields:")
print(data.dtype.names)

print("\nCapacity:")
print(data["Capacity"])

print("\nMeasurement lengths:")

for field in data.dtype.names:
    value = data[field]
    print(field, "→", value.shape)

Measurement fields:
('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time', 'Capacity')

Capacity:
[[1.85648742]]

Measurement lengths:
Voltage_measured → (1, 197)
Current_measured → (1, 197)
Temperature_measured → (1, 197)
Current_load → (1, 197)
Voltage_load → (1, 197)
Time → (1, 197)
Capacity → (1, 1)


In [24]:
discharge_cycle = cycle_data[0, discharge_index]

print("Type:")
print(discharge_cycle["type"])

print("\nAmbient temperature:")
print(discharge_cycle["ambient_temperature"])

print("\nData fields:")
print(discharge_cycle["data"].dtype.names)

Type:
['discharge']

Ambient temperature:
[[24]]

Data fields:
('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time', 'Capacity')


In [23]:
for i in range(cycle_data.shape[1]):
    cycle_type = cycle_data[0, i]["type"].item()

    if cycle_type == "discharge":
        discharge_index = i
        break

print("First discharge operation index:", discharge_index)

First discharge operation index: 1


In [22]:
first_cycle = cycle_data[0, 0]

print("Type:")
print(first_cycle["type"])

print("\nAmbient temperature:")
print(first_cycle["ambient_temperature"])

print("\nTime:")
print(first_cycle["time"])

print("\nData type:")
print(type(first_cycle["data"]))

print("\nData shape:")
print(first_cycle["data"].shape)

print("\nData fields:")
print(first_cycle["data"].dtype.names)

Type:
['charge']

Ambient temperature:
[[24]]

Time:
[[2.0080e+03 4.0000e+00 2.0000e+00 1.3000e+01 8.0000e+00 1.7921e+01]]

Data type:
<class 'numpy.ndarray'>

Data shape:
(1, 1)

Data fields:
('Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time')


In [21]:
from collections import Counter

cycle_types = []

for i in range(cycle_data.shape[1]):
    cycle_type = cycle_data[0, i]["type"]
    cycle_type = cycle_type.item()
    cycle_types.append(cycle_type)

print(Counter(cycle_types))

Counter({'impedance': 278, 'charge': 170, 'discharge': 168})


In [20]:
print("Fields:", cycle_data.dtype.names)

Fields: ('type', 'ambient_temperature', 'time', 'data')


In [ ]:
cycle_data = cycles[0, 0]

print("Type:", type(cycle_data))
print("Shape:", cycle_data.shape)
print("Dtype:", cycle_data.dtype)

In [ ]:
print("Number of cycle records:", cycles.shape[1])

In [ ]:
cycles = battery_data["B0005"]["cycle"]

print("Type:", type(cycles))
print("Shape:", cycles.shape)

In [ ]:
print(battery_data["B0005"].dtype)

In [ ]:
print(type(battery_data["B0005"]))

In [ ]:
from scipy.io import loadmat

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("B0005.mat") as f:
        battery_data = loadmat(f)

print(battery_data.keys())

In [ ]:
from scipy.io import loadmat

battery_path = project_path / "data" / "raw" / "5. Battery Data Set" / "1. BatteryAgingARC-FY08Q4.zip"

print(battery_path)

In [ ]:
import scipy
print("SciPy version:", scipy.__version__)

In [ ]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    readme = z.read("README.txt").decode("utf-8", errors="replace")

print(readme)

In [ ]:
import zipfile

zip_path = project_path / "data" / "raw" / "5. Battery Data Set" / "1. BatteryAgingARC-FY08Q4.zip"

print("ZIP file:")
print(zip_path)

print("\nFiles inside ZIP:\n")

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(name)

In [ ]:
from pathlib import Path

# Start from the BatteryIQ project folder
project_path = Path.cwd().parent

print("Project folder:")
print(project_path)

print("\nSearching for ZIP files...\n")

zip_files = list(project_path.rglob("*.zip"))

for file in zip_files:
    print(file)

In [ ]:
from pathlib import Path

dataset_path = Path(r"C:\Users\Renuka\Desktop\BatteryIQ\data\raw\Battery Data Set")

print("Does the folder exist?", dataset_path.exists())

print("\nFiles in Battery Data Set:")
for item in dataset_path.iterdir():
    print(item.name)

In [ ]:
dataset_path = raw_path / "Battery Data Set"

print("Files in Battery Data Set:\n")

for item in dataset_path.iterdir():
    print(item.name)

In [ ]:
from pathlib import Path

raw_path = Path("../data/raw")

print("Raw data location:")
print(raw_path.resolve())

print("\nContents:")
for item in raw_path.iterdir():
    print(item.name)

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

In [ ]:
import zipfile
import os

zip_path = os.path.join(
    raw_path,
    "Battery Data Set",
    "1. BatteryAgingARC-FY08Q4.zip"
)

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:")
    for name in z.namelist():
        print(name)

In [ ]:
import os

raw_path = "../data/raw"

for root, dirs, files in os.walk(raw_path):
    level = root.replace(raw_path, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    
    for file in files:
        print(f"{indent}    {file}")

In [ ]:
import os
import pandas as pd
import numpy as np

print("Environment ready!")